Semantic embedding network (COCO classes)

**Encoder: Sentence-BERT** (`sentence-transformers/all-mpnet-base-v2`)

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from matplotlib.patheffects import withStroke
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
%matplotlib inline

REPO = Path("..").resolve()
for p in (REPO, REPO / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from common import resolve_path
from src.evaluation.owod_split import COCO_CLASSES, get_split

In [ ]:

COCO_JSONS = [
    REPO / "data/OWDETR/VOC2007/Annotations/instances_train2017.json",
    REPO / "data/OWDETR/VOC2007/Annotations/instances_val2017.json",
]
OUT_DIR = REPO / "outputs/notebook/viz/semantic_network"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sentence-BERT (MPNet backbone, semantic-similarity tuned)
SENTENCE_BERT_MODEL = "sentence-transformers/all-mpnet-base-v2"

SPLIT_TASK = None          # None = all 80 COCO classes; 1-4 = OWOD known classes for that task
K_NEIGHBORS = 6            
N_CLUSTERS = 12
SEED = 0

USE_SPRING_LAYOUT = True
SPRING_ITERATIONS = 400
SPRING_K = 0.55            # lower = tighter spring layout
UMAP_NEIGHBORS = 8
UMAP_MIN_DIST = 0.02

FIGSIZE = (12, 12)
NODE_SIZE_MIN = 80
NODE_SIZE_MAX = 2200
EDGE_ALPHA = 0.65
EDGE_CURVE = 0.15
LABEL_FONTSIZE = 8
AXIS_PAD_FRAC = 0.22     # margin so edge nodes/labels are not clipped
SAVE_PAD_INCHES = 0.45   # extra border in saved PDF
OUTPUT_NAME = "semantic_network_sentence.pdf"

In [ ]:
def build_paired_palette():
    cmap = plt.get_cmap("Paired")
    n = cmap.N if hasattr(cmap, "N") else 12
    return cmap(np.linspace(0, 1, n))


def set_axes2d_equal_centered_padded(ax, xy, pad_frac=AXIS_PAD_FRAC):
    pts = np.asarray(xy, dtype=float)
    if pts.ndim != 2 or pts.shape[0] == 0 or pts.shape[1] < 2:
        return
    pts = pts[:, :2]
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    span = float((maxs - mins).max())
    if span <= 0:
        span = 1.0
    ctr = (mins + maxs) * 0.5
    half = span * 0.5 * (1.0 + pad_frac)
    ax.set_xlim(ctr[0] - half, ctr[0] + half)
    ax.set_ylim(ctr[1] - half, ctr[1] + half)
    ax.set_aspect("equal", adjustable="box")
    ax.margins(x=0.02, y=0.02)


def class_list(split_task):
    if split_task is None:
        return [COCO_CLASSES[i] for i in sorted(COCO_CLASSES)]
    return get_split(int(split_task)).known_names


def coco_category_frequencies(json_paths, names):
    id_to_name = {int(k): v for k, v in COCO_CLASSES.items()}
    name_set = set(names)
    counts = Counter()
    for jp in json_paths:
        jp = resolve_path(jp)
        if not jp.is_file():
            continue
        with open(jp) as f:
            data = json.load(f)
        cats = {c["id"]: c["name"] for c in data.get("categories", [])}
        for ann in data.get("annotations", []):
            cid = ann["category_id"]
            name = id_to_name.get(cid) or cats.get(cid)
            if name in name_set:
                counts[name] += 1
    if not counts:
        return {n: 1 for n in names}
    return {n: counts.get(n, 1) for n in names}


def class_name_prompt(name: str) -> str:
    phrase = name.replace("_", " ").strip()
    return f"a {phrase}"


def load_sentence_bert_embeddings(names, model_name):
    from sentence_transformers import SentenceTransformer

    prompts = [class_name_prompt(n) for n in names]
    model = SentenceTransformer(model_name)
    emb = model.encode(
        prompts,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=64,
    )
    return np.asarray(emb, dtype=np.float32)


def knn_edges(emb, k):
    n = emb.shape[0]
    k = min(k, n - 1)
    nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(emb)
    _, idx = nn.kneighbors(emb)
    edges = set()
    for i in range(n):
        for j in idx[i, 1:]:
            a, b = (i, int(j)) if i < j else (int(j), i)
            edges.add((a, b))
    return sorted(edges)


def cluster_labels(emb, n_clusters):
    n_clusters = min(n_clusters, emb.shape[0])
    model = AgglomerativeClustering(
        n_clusters=n_clusters,
        metric="cosine",
        linkage="average",
    )
    return model.fit_predict(emb)


def layout_positions(emb, edges, seed, use_spring=True):
    from viz import reduce_embedding

    pos = reduce_embedding(
        "umap", emb, plot_dim=2, seed=seed,
        umap_neighbors=UMAP_NEIGHBORS, umap_min_dist=UMAP_MIN_DIST,
    )
    if not use_spring:
        return pos

    g = nx.Graph()
    g.add_nodes_from(range(emb.shape[0]))
    g.add_edges_from(edges)
    init = {i: pos[i] for i in range(emb.shape[0])}
    return np.array(
        [
            nx.spring_layout(
                g,
                pos=init,
                seed=seed,
                iterations=SPRING_ITERATIONS,
                k=SPRING_K,
                weight=None,
            )[i]
            for i in range(emb.shape[0])
        ],
        dtype=float,
    )


def node_sizes(freq_map, names, vmin=NODE_SIZE_MIN, vmax=NODE_SIZE_MAX):
    f = np.array([freq_map[n] for n in names], dtype=float)
    f = np.sqrt(f)
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)
    return vmin + f * (vmax - vmin)

In [ ]:
names = class_list(SPLIT_TASK)
freq = coco_category_frequencies(COCO_JSONS, names)
emb = load_sentence_bert_embeddings(names, SENTENCE_BERT_MODEL)

edges = knn_edges(emb, K_NEIGHBORS)
communities = cluster_labels(emb, N_CLUSTERS)
pos = layout_positions(emb, edges, SEED, use_spring=USE_SPRING_LAYOUT)
sizes = node_sizes(freq, names)
palette = build_paired_palette()
node_colors = palette[communities % len(palette)]

In [ ]:
def draw_semantic_network(ax, pos, names, edges, communities, sizes, node_colors, palette):
    for i, j in edges:
        c = palette[communities[i] % len(palette)]
        x1, y1 = pos[i]
        x2, y2 = pos[j]
        rad = EDGE_CURVE if (i + j) % 2 == 0 else -EDGE_CURVE
        ax.annotate(
            "",
            xy=(x2, y2),
            xytext=(x1, y1),
            arrowprops=dict(
                arrowstyle="-",
                color=(*c[:3], EDGE_ALPHA),
                lw=0.9,
                shrinkA=6,
                shrinkB=6,
                connectionstyle=f"arc3,rad={rad}",
            ),
            zorder=1,
        )

    ax.scatter(
        pos[:, 0],
        pos[:, 1],
        s=sizes,
        c=node_colors,
        edgecolors="0.5",
        linewidths=0.4,
        alpha=1.0,
        zorder=3,
    )

    for i, name in enumerate(names):
        t = ax.text(
            pos[i, 0],
            pos[i, 1],
            name,
            ha="center",
            va="center",
            fontsize=LABEL_FONTSIZE,
            fontweight="medium",
            color="0.15",
            zorder=4,
        )
        t.set_path_effects([withStroke(linewidth=2.5, foreground="white")])

    set_axes2d_equal_centered_padded(ax, pos)
    ax.set_axis_off()


fig, ax = plt.subplots(figsize=FIGSIZE, facecolor="white")
draw_semantic_network(ax, pos, names, edges, communities, sizes, node_colors, palette)
title = "COCO semantic word network (Sentence-BERT)"
if SPLIT_TASK is not None:
    title += f" — OWOD task {SPLIT_TASK}"
ax.set_title(title, fontsize=14, pad=12)
fig.subplots_adjust(left=0.02, right=0.98, top=0.94, bottom=0.02)

out_path = OUT_DIR / OUTPUT_NAME
fig.savefig(
    out_path,
    format="pdf",
    bbox_inches="tight",
    pad_inches=SAVE_PAD_INCHES,
    facecolor="white",
    edgecolor="none",
)
plt.show()